# Tutorial 4: LLM Diagnostics

Use comfio's LLM-native tools for building diagnostic agents.

**Requires**: `pip install comfio[agent]`

In [ ]:
import numpy as np
from comfio import (
    evaluate_thermal, evaluate_visual, evaluate_acoustic, evaluate_iaq,
    calculate_global_ieq, ieq_to_markdown, ieq_to_summary_dict,
    EDGE_SYSTEM_PROMPT, DIAGNOSTIC_PROMPT_TEMPLATE, format_prompt,
)

## 1. Run IEQ Evaluation

In [ ]:
rng = np.random.default_rng(42)
n = 50

thermal = evaluate_thermal(
    tdb=rng.normal(25.0, 2.0, n),
    tr=rng.normal(25.0, 1.5, n),
    vr=np.full(n, 0.1),
    rh=rng.normal(55.0, 5.0, n),
    met=1.2, clo=0.5,
)
visual = evaluate_visual(illuminance=rng.normal(450.0, 100.0, n))
acoustic = evaluate_acoustic(laeq=rng.normal(45.0, 8.0, n))
iaq = evaluate_iaq(co2=rng.normal(900.0, 150.0, n))

ieq = calculate_global_ieq(thermal=thermal, visual=visual, acoustic=acoustic, iaq=iaq)

## 2. Generate Markdown Diagnostic Report

In [ ]:
markdown = ieq_to_markdown(ieq)
print(markdown)

## 3. Generate Compact Summary Dict

In [ ]:
summary = ieq_to_summary_dict(ieq)
for key, value in summary.items():
    print(f'{key}: {value}')

## 4. Get Tool Schemas for LLM Function Calling

In [ ]:
from comfio.llm.tools import to_openai_tools, to_langchain_tools

# OpenAI format
openai_tools = to_openai_tools()
print(f'Available tools: {len(openai_tools)}')
for tool in openai_tools:
    print(f'  - {tool["function"]["name"]}')

## 5. Build a Diagnostic Prompt

In [ ]:
prompt = format_prompt(
    DIAGNOSTIC_PROMPT_TEMPLATE,
    thermal_status=f'PMV={np.mean(thermal.pmv):.2f}, PPD={np.mean(thermal.ppd):.1f}%',
    iaq_status=f'CO2={np.mean(iaq.co2):.0f} ppm',
    pollutant_status='PM2.5=12 μg/m³',
    adaptive_status='Within 80% band',
    tsv_status='Mean TSV=0.5, compliance=90%',
)

messages = [
    {'role': 'system', 'content': EDGE_SYSTEM_PROMPT},
    {'role': 'user', 'content': prompt},
]
print('System prompt length:', len(EDGE_SYSTEM_PROMPT), 'chars')
print('Diagnostic prompt length:', len(prompt), 'chars')